# Standard ChEBI SFT On Kaggle

This notebook ensures the ChEBI-20 processed splits exist, stays compatible with the upstream diverse-beam collection stage artifact chain, runs standard single-molecule SFT, and exports a reusable artifact bundle with checkpoints and summaries.


In [ ]:
from pathlib import Path
import sys

REPO_URL = "https://github.com/mruniverse8/Thesis.git"
REPO_BRANCH = "gflownet"
REPO_DIR = Path("/kaggle/working/Thesis")
STAGE_NAME = "train_sft"
UPSTREAM_COLLECTION_STAGE = "collect_chebi_biot5"

%cd /kaggle/working
!if [ -d "{REPO_DIR / '.git'}" ]; then echo "Reusing {REPO_DIR}"; elif [ -d "{REPO_DIR}" ]; then echo "Existing non-git directory at {REPO_DIR}; delete it and rerun the notebook." && false; else git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"; fi
!git -C "{REPO_DIR}" fetch --depth 1 origin "{REPO_BRANCH}" && git -C "{REPO_DIR}" checkout -B "{REPO_BRANCH}" FETCH_HEAD

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from thesis_kaggle_support import (
    copy_stage_artifact_to_local,
    ensure_paths_exist,
    ensure_repo_selfies_vocab,
    ensure_runtime_dependencies,
    dump_yaml,
    export_stage_artifacts,
    json_dumps,
    load_yaml,
    read_json,
    report_runtime,
)


In [ ]:
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
NUM_EPOCHS = 1
NUM_WORKERS = 2
CHEBI_OUTPUT_DIR = REPO_DIR / "data" / "chebi20"
OUTPUT_DIR = REPO_DIR / "outputs" / "kaggle" / "chebi20_sft"
TEMP_CONFIG_PATH = REPO_DIR / "kaggle" / "generated_configs" / "sft_chebi20.kaggle.yaml"

ensure_runtime_dependencies(REPO_DIR)
runtime_report = report_runtime(require_gpu=True)
print(json_dumps({
    "runtime": runtime_report,
    "output_dir": str(OUTPUT_DIR),
}))


In [ ]:
CHEBI_PROCESSED_DIR = CHEBI_OUTPUT_DIR / "processed"
if not all((CHEBI_PROCESSED_DIR / f"{split}.jsonl").exists() for split in ("train", "validation", "test")):
    copied_dir = copy_stage_artifact_to_local(
        stage_name=UPSTREAM_COLLECTION_STAGE,
        artifact_relpath="chebi20_processed",
        local_path=CHEBI_PROCESSED_DIR,
    )
else:
    copied_dir = None

have_processed = all((CHEBI_PROCESSED_DIR / f"{split}.jsonl").exists() for split in ("train", "validation", "test"))
print(json_dumps({
    "copied_dir": None if copied_dir is None else str(copied_dir),
    "have_processed": have_processed,
    "processed_dir": str(CHEBI_PROCESSED_DIR),
}))

%cd {REPO_DIR}
!if [ -f "{CHEBI_PROCESSED_DIR / 'train.jsonl'}" ] && [ -f "{CHEBI_PROCESSED_DIR / 'validation.jsonl'}" ] && [ -f "{CHEBI_PROCESSED_DIR / 'test.jsonl'}" ]; then echo "ChEBI processed splits ready"; else python scripts/download_chebi20.py --output-dir "{CHEBI_OUTPUT_DIR}"; fi

config = load_yaml(REPO_DIR / "configs" / "sft_chebi20.yaml")
config["data"]["train_file"] = str(CHEBI_PROCESSED_DIR / "train.jsonl")
config["data"]["validation_file"] = str(CHEBI_PROCESSED_DIR / "validation.jsonl")
config["data"]["test_file"] = str(CHEBI_PROCESSED_DIR / "test.jsonl")
config["data"]["num_workers"] = int(NUM_WORKERS)
config["training"]["output_dir"] = str(OUTPUT_DIR)
config["training"]["per_device_train_batch_size"] = int(PER_DEVICE_TRAIN_BATCH_SIZE)
config["training"]["per_device_eval_batch_size"] = int(PER_DEVICE_EVAL_BATCH_SIZE)
config["training"]["gradient_accumulation_steps"] = int(GRADIENT_ACCUMULATION_STEPS)
config["training"]["num_epochs"] = int(NUM_EPOCHS)
dump_yaml(config, TEMP_CONFIG_PATH)
print(f"Wrote config: {TEMP_CONFIG_PATH}")
print(TEMP_CONFIG_PATH.read_text(encoding="utf-8"))


In [ ]:
%cd {REPO_DIR}
!python scripts/train_sft.py --config "{TEMP_CONFIG_PATH}"


In [ ]:
required_outputs = ensure_paths_exist({
    "output_dir": OUTPUT_DIR,
    "best_checkpoint": OUTPUT_DIR / "checkpoints" / "best",
    "run_summary": OUTPUT_DIR / "run_summary.json",
    "history": OUTPUT_DIR / "history.json",
    "resolved_config": OUTPUT_DIR / "resolved_config.yaml",
    "tokenizer_metadata": OUTPUT_DIR / "tokenizer_metadata.json",
})
artifact_dir, manifest = export_stage_artifacts(
    stage_name=STAGE_NAME,
    artifact_map={
        "checkpoints": OUTPUT_DIR / "checkpoints",
        "run_summary.json": OUTPUT_DIR / "run_summary.json",
        "history.json": OUTPUT_DIR / "history.json",
        "resolved_config.yaml": OUTPUT_DIR / "resolved_config.yaml",
        "tokenizer_metadata.json": OUTPUT_DIR / "tokenizer_metadata.json",
    },
    metadata={
        "required_outputs": required_outputs,
        "config_path": str(TEMP_CONFIG_PATH),
        "summary": read_json(OUTPUT_DIR / "run_summary.json"),
    },
)
print(json_dumps({
    "artifact_dir": str(artifact_dir),
    "manifest": manifest,
}))
